# Bronze Ingestion: BC Soil Information (DataBC WFS)

Pulls the soils data behind the **BC Soil Information Finder Tool (SIFT)** from the **BC Geographic Warehouse (DataBC) WFS**, clipped to the pipeline-selected AOI bounding box.

| Bronze table | DataBC layer | Theme |
| --- | --- | --- |
| `bronze_bc_soil_survey_polygons` | `STE_SOIL_SURVEYS_MVW` | Soil survey polygons |
| `bronze_bc_soil_project_boundaries` | `STE_SOIL_PROJ_BOUNDARIES_SVW` | Soil survey project boundaries |

Source: [BC Soil Information Finder Tool](https://www2.gov.bc.ca/gov/content/environment/air-land-water/land/soil/soil-information-finder).

Features are fetched as GeoJSON and stored raw as geometry and properties JSON. These are British Columbia sources. An AOI outside source coverage can produce empty tables, which downstream reporting must identify as a data gap rather than a measured zero.

## 1. Parameters

In [ ]:
# Pipeline parameters
LATITUDE = 49.2193
LONGITUDE = -122.5984
RADIUS_KM = 20

## 2. Helpers — AOI bbox + WFS fetch

In [ ]:
import math
import json
import requests
from datetime import datetime, timezone

if not -90.0 <= float(LATITUDE) <= 90.0:
    raise ValueError("LATITUDE must be between -90 and 90 degrees.")
if not -180.0 <= float(LONGITUDE) <= 180.0:
    raise ValueError("LONGITUDE must be between -180 and 180 degrees.")
if not 0.0 < float(RADIUS_KM) <= 100.0:
    raise ValueError("RADIUS_KM must be greater than 0 and no more than 100 km.")

AOI_NAME = f"AOI {float(LATITUDE):.4f}, {float(LONGITUDE):.4f}"
WFS_ENDPOINT = "https://openmaps.gov.bc.ca/geo/pub/wfs"
MAX_FEATURES = 5000
LAYERS = {
    "bronze_bc_soil_survey_polygons": "pub:WHSE_TERRESTRIAL_ECOLOGY.STE_SOIL_SURVEYS_MVW",
    "bronze_bc_soil_project_boundaries": "pub:WHSE_TERRESTRIAL_ECOLOGY.STE_SOIL_PROJ_BOUNDARIES_SVW",
}


def radius_to_bbox(lat, lon, radius_km):
    """Approximate a WGS84 bounding box around a point."""
    lat_delta = radius_km / 111.32
    cos_lat = math.cos(math.radians(lat))
    lon_delta = 180.0 if abs(cos_lat) < 1e-8 else radius_km / (111.32 * cos_lat)
    return [lon - lon_delta, lat - lat_delta, lon + lon_delta, lat + lat_delta]


BBOX = radius_to_bbox(float(LATITUDE), float(LONGITUDE), float(RADIUS_KM))
# WFS 2.0 bbox order is miny,minx,maxy,maxx (lat/lon) followed by the CRS URN.
BBOX_PARAM = f"{BBOX[1]},{BBOX[0]},{BBOX[3]},{BBOX[2]},urn:ogc:def:crs:EPSG::4326"
print({"aoi": AOI_NAME, "radius_km": RADIUS_KM, "tables": list(LAYERS)})
print("AOI bbox [minlon, minlat, maxlon, maxlat]:", BBOX)


def fetch_wfs_features(layer, bbox_param, max_features):
    params = {
        "service": "WFS",
        "version": "2.0.0",
        "request": "GetFeature",
        "typeNames": layer,
        "count": str(max_features),
        "outputFormat": "application/json",
        "srsName": "EPSG:4326",
        "bbox": bbox_param,
    }
    response = requests.get(WFS_ENDPOINT, params=params, timeout=180)
    response.raise_for_status()
    return response.json().get("features", [])

## 3. Ingestion function — GeoJSON features to bronze Delta

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType,
)

BRONZE_SCHEMA = StructType([
    StructField("feature_id", StringType(), True),
    StructField("source_layer", StringType(), True),
    StructField("geometry_type", StringType(), True),
    StructField("geometry_json", StringType(), True),
    StructField("properties_json", StringType(), True),
    StructField("aoi_name", StringType(), True),
    StructField("aoi_lat", DoubleType(), True),
    StructField("aoi_lon", DoubleType(), True),
    StructField("aoi_radius_km", DoubleType(), True),
    StructField("ingested_at", TimestampType(), True),
])


def ingest_layer(table_name, layer):
    feats = fetch_wfs_features(layer, BBOX_PARAM, MAX_FEATURES)
    print(f"{table_name}: {len(feats)} features fetched from {layer}")
    now = datetime.now(timezone.utc)
    rows = []
    for f in feats:
        geom = f.get("geometry") or {}
        rows.append((
            str(f.get("id")),
            layer,
            geom.get("type"),
            json.dumps(geom),
            json.dumps(f.get("properties", {})),
            AOI_NAME,
            float(LATITUDE),
            float(LONGITUDE),
            float(RADIUS_KM),
            now,
        ))
    if not rows:
        print(f"WARNING: no features for {table_name}; writing empty table with schema")
    df = spark.createDataFrame(rows, schema=BRONZE_SCHEMA)
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(table_name))
    return df.count()

## 4. Run ingestion for all layers

In [ ]:
results = {}
for table_name, layer in LAYERS.items():
    results[table_name] = ingest_layer(table_name, layer)

print("\nIngestion complete:")
for t, c in results.items():
    print(f"  {t}: {c} rows")

## 5. Verify bronze tables

In [ ]:
for table_name in LAYERS:
    cnt = spark.table(table_name).count()
    print(f"{table_name}: {cnt} rows")
    display(spark.table(table_name).limit(3))